# Tamil call transcription - batch run

Dual-T4 batched pipeline over attached `tamil-calls-*` datasets.
Uses pre-converted CT2 model (`tamil-ct2-model`) + pipeline code
(`voice-processing-code`). Outputs: `out/transcripts/*.json|.txt` +
`out/manifest.gpu*.jsonl` (resumable).

In [ ]:
!nvidia-smi -L
!pip -q install faster-whisper ctranslate2 transformers soundfile

In [ ]:
import glob, os, json

MODEL_DIR = '/kaggle/input/tamil-ct2-model'
CODE = '/kaggle/input/voice-processing-code/transcribe.py'
AUDIO_DIRS = sorted(d for d in glob.glob('/kaggle/input/*')
                    if os.path.basename(d).startswith('tamil-calls'))

assert os.path.isfile(os.path.join(MODEL_DIR, 'model.bin')), 'attach tamil-ct2-model'
assert os.path.isfile(CODE), 'attach voice-processing-code'
assert AUDIO_DIRS, 'attach at least one tamil-calls-* dataset'

n = sum(len([f for f in os.listdir(d) if not f.endswith('.json')]) for d in AUDIO_DIRS)
print('model ok | code ok | audio dirs:', AUDIO_DIRS)
print('audio files to process:', n)

In [ ]:
import subprocess, sys

args = [sys.executable, CODE]
for d in AUDIO_DIRS:
    args += ['--input-dir', d]
args += ['--output-dir', '/kaggle/working/out',
         '--model', MODEL_DIR,
         '--workers', '2', '--device', 'cuda',
         '--batch-size', '16', '--beam-size', '5',
         '--compute-type', 'int8_float16', '--language', 'ta']

print(' '.join(args))
r = subprocess.run(args)
print('exit', r.returncode)

In [ ]:
import glob, json, os
txts = sorted(glob.glob('/kaggle/working/out/transcripts/*.txt'))
print(f'transcripts written: {len(txts)}')
errs = []
for m in glob.glob('/kaggle/working/out/manifest*.jsonl'):
    for line in open(m, encoding='utf-8'):
        r = json.loads(line)
        if r.get('status') != 'ok': errs.append(r)
print('errors:', len(errs))
for e in errs[:10]: print(' ', os.path.basename(e['file']), e.get('error'))
if txts:
    print('='*60)
    print(os.path.basename(txts[0]))
    print(open(txts[0], encoding='utf-8').read()[:800])